<a href="https://colab.research.google.com/github/pakizahassan/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pakizahassan/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


### Feature Vector Design

I built one page-level feature vector from the March 2026 FlyRank content performance data.

The feature vector uses historical search and engagement signals. Anonymous identifiers are retained only for grouping and traceability; they are not used as predictive features.

The engineered features include daily impressions, CTR, average search position, daily sessions, engagement rate, daily scroll activity, and GA4 data coverage.

Missing GA4 activity is represented together with a coverage feature so that unavailable analytics data is not silently treated as equivalent to complete measurement.

In [6]:

import pandas as pd
import numpy as np

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "ga4_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/data_0.parquet",
    columns=columns
)

df["report_date"] = pd.to_datetime(df["report_date"])

print("Rows loaded:", len(df))
print("Unique pages:", df["content_hash_id"].nunique())
print("Unique clients:", df["client_hash_id"].nunique())

Rows loaded: 9841378
Unique pages: 331437
Unique clients: 55


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
feature_df = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        observed=True
    )
    .agg(
        observed_days=("report_date", "nunique"),
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        sessions=("ga4_sessions", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        scroll_events=("scroll_events", "sum"),
        ga4_available_days=("ga4_data_available", "sum")
    )
    .reset_index()
)

# Engineered features
feature_df["daily_impressions"] = (
    feature_df["impressions"] /
    feature_df["observed_days"]
)

feature_df["ctr"] = np.where(
    feature_df["impressions"] > 0,
    feature_df["clicks"] / feature_df["impressions"],
    0
)

feature_df["daily_sessions"] = (
    feature_df["sessions"] /
    feature_df["observed_days"]
)

feature_df["engagement_rate"] = np.where(
    feature_df["sessions"] > 0,
    feature_df["engaged_sessions"] / feature_df["sessions"],
    0
)

feature_df["daily_scroll_events"] = (
    feature_df["scroll_events"] /
    feature_df["observed_days"]
)

feature_df["ga4_coverage"] = (
    feature_df["ga4_available_days"] /
    feature_df["observed_days"]
)

# Preserve missing-position information
feature_df["position_missing"] = (
    feature_df["avg_position"].isna().astype("int8")
)

position_fill = feature_df["avg_position"].median()

feature_df["avg_position"] = (
    feature_df["avg_position"]
    .fillna(position_fill)
)

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
model_features = [
    "daily_impressions",
    "ctr",
    "avg_position",
    "daily_sessions",
    "engagement_rate",
    "daily_scroll_events",
    "ga4_coverage",
    "position_missing"
]

X = (
    feature_df[model_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

print("Feature-vector rows:", len(X))
print("Number of model features:", X.shape[1])
print("Missing values remaining:", int(X.isna().sum().sum()))

display(X.head())


Feature-vector rows: 331437
Number of model features: 8
Missing values remaining: 0


/tmp/ipykernel_3843/372496875.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace([np.inf, -np.inf], np.nan)


,daily_impressions,ctr,avg_position,daily_sessions,engagement_rate,daily_scroll_events,ga4_coverage,position_missing
0,0.000000,0.0,8.505296,0.0,0.0,0.0,0.0,1.0
1,0.000000,0.0,8.505296,0.0,0.0,0.0,0.0,1.0
2,0.000000,0.0,8.505296,0.0,0.0,0.0,0.0,1.0
3,0.032258,0.0,9.000000,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,8.505296,0.0,0.0,0.0,0.0,1.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



### Excluded Fields

I deliberately excluded fields that could create memorization, privacy risk, or leakage.

- `client_hash_id` — retained only for grouping. It is an anonymous identifier and should not be used as a predictive signal.
- `content_hash_id` — retained only to identify anonymous pages. Using it as a model feature could encourage page-level memorization.
- `report_date` — used to define observation windows, but not used directly as a predictive feature.
- Future-period performance metrics — excluded because they would not be available at the moment a prediction is made.
- Target-derived fields — excluded because using information directly involved in creating the prediction target would create circularity.
- Product-generated recommendation or scoring fields — excluded if present because they may contain existing business logic or downstream information rather than independent raw evidence.

The final feature vector therefore uses only historical, public-safe search and engagement measurements that are available before the prediction point.

In [9]:
excluded_fields = pd.DataFrame({
    "Excluded Field": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "future-period metrics",
        "target-derived fields",
        "product-generated scores/actions"
    ],
    "Reason": [
        "Anonymous client ID; grouping only, not a predictive feature",
        "Anonymous page ID; avoids page-level memorization",
        "Used for time/window logic, not as a model signal",
        "Would leak information unavailable at prediction time",
        "Would create circularity with the prediction target",
        "May contain downstream business logic or existing recommendations"
    ]
})

display(excluded_fields)

,Excluded Field,Reason
0,client_hash_id,"Anonymous client ID; grouping only, not a pred..."
1,content_hash_id,Anonymous page ID; avoids page-level memorization
2,report_date,"Used for time/window logic, not as a model signal"
3,future-period metrics,Would leak information unavailable at predicti...
4,target-derived fields,Would create circularity with the prediction t...
5,product-generated scores/actions,May contain downstream business logic or exist...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.